In [1]:
#Importing required libraries
import matplotlib.pyplot as plt
import random
import copy

In [2]:
#Game State definition:
state = {
    "player_1": [],
    "player_2": [],
    "player_3": [],
    "top_card": [],
    "turn": [],
    "deck": []
}

In [3]:
#Card class definition
class Card:
    def __init__(self, color, value):

        self.color = color
        self.value = value

    def __repr__(self):

        return f"{self.color} {self.value}"

    def __eq__(self, other):

        if not isinstance(other, Card):
            return False

        return self.color == other.color and self.value == other.value

    def __hash__(self):

        return hash((self.color, self.value))

In [4]:
#Generating Deck Of Cards
def generate_deck():

  #Defining colors
  colors = ["Red", "Blue", "Green", "Yellow"]
  deck = []

  #Every number + Skip for each color
  for color in colors:
    for i in range(1, 10):
      deck.append(Card(color, i))
    deck.append(Card(color, "Skip")) #Adding Skip

  #Shuffling using random
  random.shuffle(deck)

  return deck

In [5]:
#Dealing Cards as list, 5 each:
def deal_cards(deck):

  return [deck.pop() for _ in range(5)]

In [6]:
# Terminal State / game over (one player wins)
def is_terminal(state):

  return (len(state["p1"]) == 0 or len(state["p2"]) == 0 or len(state["p3"]) == 0)

In [7]:
#Evaluation Function: Score = 50 − 5(CAI) + 2(Copp) + 3(S)
def evaluate(state, player):

  #CAI
  my_cards = len(state[player])

  #Copp
  oppenents = [x for x in ["p1", "p2", "p3"] if x != player]
  opp_cards = 0
  for opp in oppenents:
    opp_cards += len(state[opp])

  #S
  skip_cards = sum(1 for c in state[player] if c.value == "Skip")

  score = 50 - 5 * my_cards + 2 * opp_cards + 3 * skip_cards

  return score


In [8]:
#Valid Move Check (Returns list of valid moves)
def valid_moves(hand, top_card):

  valid = []

  #Appending valid moves
  for card in hand:

    if card.color == top_card.color or card.value == top_card.value: #Validation Check
      valid.append(card)

  #If no valid moves then draw from deck
  if not valid:

    return ["DRAW"]

  return valid

In [9]:
#Evaluation Function: Score = 50 − 5(CAI) + 2(Copp) + 3(S)
def evaluate(state, player):

  #CAI
  my_cards = len(state[player])

  #Copp
  oppenents = [x for x in ["p1", "p2", "p3"] if x != player]
  opp_cards = 0
  for opp in oppenents:
    opp_cards += len(state[opp])

  #S
  skip_cards = sum(1 for c in state[player] if c.value == "Skip")

  score = 50 - 5 * my_cards + 2 * opp_cards + 3 * skip_cards

  return score

In [10]:
# State Transition : apply move function:
def apply_move(state, player, move):

  new_state = copy.deepcopy(state)

  #Handling Draw:
  if move == "DRAW":
    new_state[player].append(new_state["deck"].pop())

    #Next player's turn
    new_state["turn"] = (new_state["turn"] + 1) % 3
    return new_state

  #DEBUG
  if move not in new_state[player]:
    print("ERROR: Move not in hand!")
    print("Move:", move)
    print("Hand:", new_state[player])
    raise Exception("Invalid move")

  #Playing Card
  new_state[player].remove(move)
  new_state["top_card"] = move

  #Handling Skip:
  if move.value == "Skip":
    new_state["turn"] = (new_state["turn"] + 2) % 3
  else:
    #Normal Next Turn:
    new_state["turn"] = (new_state["turn"] + 1) % 3

  return new_state

In [11]:
#Initializing Game
def initialize():

  #Generating Deck
  deck = generate_deck()

  #Initializing State
  state = {

    "p1": deal_cards(deck),
    "p2": deal_cards(deck),
    "p3": deal_cards(deck),
    "top_card": deck.pop(),
    "turn": 0,
    "deck": deck
  }

  return state

In [12]:
#Printer Function:
def print_state(state):

    print("\n---------------------------------------------\n")

    print(f"Top Card: {state['top_card']}")

    print("\nP1 (Minimax) Cards:")
    print([str(c) for c in state["p1"]])

    print("\nP2 (Expectimax) Cards:")
    print([str(c) for c in state["p2"]])

    print("\nP3 Cards:")
    print([str(c) for c in state["p3"]])

    print("-----------------------------------------------\n")

In [13]:
# Get current player from turn
def get_current_player(state):

  if state["turn"] == 0:

    return "p1"

  elif state["turn"] == 1:

    return "p2"

  elif state["turn"] == 2:

    return "p3"

In [16]:
#expectimax evaluator:
def expectimax(state, depth, player):

  if depth == 0 or is_terminal(state):
    return evaluate(state, player)

  current = get_current_player(state)
  moves = valid_moves(state[current], state["top_card"])

  if current == player:
    best_score = -float("inf")

    for move in moves:
      new_state = apply_move(state, current, move)
      val = expectimax(new_state, depth - 1, player)
      best_score = max(best_score, val)

    return best_score

  #Draw Case
  if moves == ["Draw"]:

    deck_size = len(state["deck"])

    if deck_size == 0:
      return evaluate(state, player)

    total_score = 0

    for card in state["deck"]:

      prob = 1 / deck_size
      new_state = copy.deepcopy(state)
      new_state[current].append(card)

      val = expectimax(new_state, depth - 1, player)
      total_score += val * prob

    return total_score

  #Opponent
  total = 0
  prob = 1 / len(moves)

  for move in moves:
    new_state = apply_move(state, current, move)
    val = expectimax(new_state, depth - 1, player)
    total += val * prob

  return total

In [17]:
#Best move Using Expectimax:
def best_move_expectimax(state, player, depth = 3):

  #Varibales
  best_score = -float("inf")
  best_move = None
  hand = state[player]
  top_card = state["top_card"]

  #Get Valid Moves
  moves = valid_moves(hand, top_card)

  print("\nExpectimax Decision: ")
  for move in moves:

    new_state = apply_move(state, player, move)
    score = expectimax(new_state, depth - 1, player)

    print(f"Move: {move}, Score: {score}")

    if score > best_score:

      best_score = score
      best_move = move

  return best_move

In [18]:
def minimax(state, depth, maximizing, player):

  #Evaluate
  if depth == 0 or is_terminal(state):

    return evaluate(state, player)

  current = get_current_player(state)
  moves = valid_moves(state[current], state["top_card"])

  if current == player:

    best_score = -float("inf")

    for move in moves:

      new_state = apply_move(state, current, move)
      val = minimax(new_state, depth - 1, False, player)
      best_score = max(best_score, val)

    return best_score

  else:

    best = float("inf")

    for move in moves:

      new_state = apply_move(state, current, move)
      val = minimax(new_state, depth - 1, True, player)
      best = min(best, val)

    return best

In [19]:
#Best move using Minimax:
def best_move_minimax(state, player, depth = 3):

  #Variables
  best_score = -float("inf")
  best_move = None
  hand = state[player]
  top_card = state["top_card"]


  #Get Valid Moves
  moves = valid_moves(hand, top_card)

  print("\nMinimax Decisions: ")

  for move in moves:

    #Apply Move,
    new_state = apply_move(state, player, move)
    score = minimax(new_state, depth - 1, False, player)
    print(f"Move: {move}, Score: {score}")

    if score > best_score:
      best_score = score
      best_move = move

  return best_move

In [20]:
#Player 3 Mode: Manual / AI
def get_p3_move(state, mode="manual"):

    hand = state["p3"]
    top = state["top_card"]

    moves = valid_moves(hand, top)

    if mode == "ai":
        return best_move_minimax(state, "p3")

    # Manual mode
    print("\nYour Turn (P3)")
    print("Top Card:", top)

    for i, card in enumerate(hand):
        print(f"{i}: {card}")

    if moves == ["DRAW"]:
        print("No valid moves. Drawing...")
        return "DRAW"

    while True:
        try:
            choice = int(input("Enter index of card to play: "))

            if 0 <= choice < len(hand):
                selected = hand[choice]

                if selected in moves:
                  return selected

                else:
                  print("Invalid move. Please select a valid card.")

            else:
                print("Invalid index. Please enter a number between 0 and", len(hand) - 1)
        except ValueError:
            print("Invalid input. Please enter a number.")